# 📓 Notebook 02 — Network, Infra & Active Directory

Host discovery, service enum, SMB/NetBIOS, credential attacks, lateral movement, exploit delivery.

# 🧱 BASE (shared setup) — run these first, in order

In [ ]:
# Which GPU did you actually get? Decide the model from this.
!nvidia-smi
import torch, psutil
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM {p.total_memory/1e9:.1f} GB | RAM {psutil.virtual_memory().total/1e9:.1f} GB")
!df -h /content | tail -1

Wed Oct 14 07:35:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

GPU: Tesla T4 | VRAM 15.6 GB | RAM 13.6 GB
overlay         236G   72G  165G  31% /


In [ ]:
import os
try:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("cache:", CACHE)
except Exception as e:
    print("Drive notice:", e)
    CACHE = '/content/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("local cache:", CACHE)


cache: /content/drive/MyDrive/redteam


In [ ]:
%%bash
apt-get -qq update
apt-get -qq install -y docker.io git curl wget jq unzip tmux nodejs npm python3-venv python3-pip net-tools iputils-ping dnsutils
# Colab blocks Docker's iptables hooks -> disable them
dockerd --iptables=false --ip6tables=false --bridge=none >/var/log/dockerd.log 2>&1 &
for i in $(seq 1 30); do docker info >/dev/null 2>&1 && break; sleep 2; done
docker info | grep -E "Server Version|Storage Driver"
echo "docker ready" 

Server Version: 29.1.3-colab
Storage Driver: vfs
docker ready


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh || true
command -v ollama || { curl -L https://ollama.com/download/ollama-linux-amd64.tgz | tar -xz -C /usr; }
nohup ollama serve >/var/log/ollama.log 2>&1 &
for i in $(seq 1 40); do curl -s localhost:11434/api/version >/dev/null && break; sleep 2; done
curl -s localhost:11434/api/version && echo " <- ollama up" 

/usr/local/bin/ollama
{"version":"0.40.4"} <- ollama up


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
%%bash
if ollama list | grep -q "qwen2.5-coder:7b"; then
  echo "qwen2.5-coder:7b ready"
else
  ollama pull qwen2.5-coder:7b
fi
ollama list


qwen2.5-coder:7b ready
NAME                ID              SIZE      MODIFIED       
gpt-oss:20b         f38aa0c53da5    13 GB     15 minutes ago    
qwen2.5-coder:7b    dae161e27b0e    4.7 GB    36 minutes ago    


In [ ]:
%%bash
# HexStrike = 150+ tools behind one MCP endpoint
if [ ! -d /content/hexstrike ]; then
  git clone --depth 1 https://github.com/0x4m4/hexstrike-ai /content/hexstrike
fi
cd /content/hexstrike
if ! curl -sf localhost:8888/health >/dev/null; then
  nohup python3 hexstrike_server.py --port 8888 >/var/log/hexstrike.log 2>&1 &
  sleep 5
fi
echo "hexstrike:"; curl -s localhost:8888/health; echo

# Kali toolbox container the agent can shell into
docker pull -q kalilinux/kali-rolling || true
docker run -d --name redteam --network host kalilinux/kali-rolling sleep infinity || true
echo toolbox-ready


hexstrike:
{"all_essential_tools_available":false,"cache_stats":{"evictions":0,"hit_rate":"8.0%","hits":122,"max_size":1000,"misses":1402,"size":12},"category_stats":{"additional":{"available":0,"total":14},"api":{"available":1,"total":8},"binary":{"available":3,"total":13},"cloud":{"available":0,"total":10},"essential":{"available":0,"total":8},"exploitation":{"available":0,"total":3},"forensics":{"available":2,"total":16},"network":{"available":0,"total":10},"osint":{"available":1,"total":13},"password":{"available":0,"total":5},"vuln_scanning":{"available":1,"total":4},"web_security":{"available":4,"total":19},"wireless":{"available":0,"total":4}},"message":"HexStrike AI Tools API Server is operational","status":"healthy","telemetry":{"average_execution_time":"0.00s","commands_executed":1402,"success_rate":"0.9%","system_metrics":{"cpu_percent":59.2,"disk_usage":30.4,"memory_percent":26.5,"network_io":{"bytes_recv":28969896209,"bytes_sent":114794241,"dropin":0,"dropout":0,"errin":0,

In [ ]:
%%bash
if ! pgrep -f "cloudflared tunnel --url http://localhost:8888" >/dev/null; then
  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared 2>/dev/null || true
  chmod +x /usr/local/bin/cloudflared 2>/dev/null || true
  nohup cloudflared tunnel --url http://localhost:8888 >/var/log/cf.log 2>&1 &
  sleep 10
fi
echo "HexStrike URL:"; grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /var/log/cf.log | head -1


HexStrike URL:
https://modification-similarly-retention-theatre.trycloudflare.com


In [ ]:
%%bash
apt-get -qq update && apt-get -qq install -y nmap masscan arp-scan hydra john hashcat smbmap nbtscan whois || true
pip -q install netexec impacket enum4linux-ng 2>/dev/null || true
[ ! -d /content/vulhub ] && git clone --depth 1 https://github.com/vulhub/vulhub /content/vulhub || true
echo "net tools ready"


(Reading database ... 159708 files and directories currently installed.)
Preparing to unpack .../libperl5.38t64_5.38.2-3.2ubuntu0.6_amd64.deb ...
Unpacking libperl5.38t64:amd64 (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2ubuntu0.2) ...
Preparing to unpack .../perl_5.38.2-3.2ubuntu0.6_amd64.deb ...
Unpacking perl (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2ubuntu0.2) ...
Preparing to unpack .../perl-base_5.38.2-3.2ubuntu0.6_amd64.deb ...
Unpacking perl-base (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2ubuntu0.2) ...
Setting up perl-base (5.38.2-3.2ubuntu0.6) ...
(Reading database ... 159708 files and directories currently installed.)
Preparing to unpack .../000-perl-modules-5.38_5.38.2-3.2ubuntu0.6_all.deb ...
Unpacking perl-modules-5.38 (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2ubuntu0.2) ...
Selecting previously unselected package python3-pygments.
Preparing to unpack .../001-python3-pygments_2.17.2+dfsg-1_all.deb ...
Unpacking python3-pygments (2.17.2+dfsg-1) ...
Selecting previously unselected package pyt

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Cloning into '/content/vulhub'...
Updating files: 100% (2900/2900), done.


In [ ]:
%%bash
# --- targets ---
docker run -d --name metasploitable2 --network host tleemcjr/metasploitable2
# SMB share host
docker run -d --name smb --network host dperson/samba -p -u "user;pass" -s "share;/tmp;yes;no;no;user"
# vulnerable FTP (vsftpd 2.3.4 backdoor)
docker run -d --name ftp --network host fauria/vsftpd
sleep 20
nmap -sV -T4 localhost -p 21,22,23,80,139,445,3306,5432,8080 | tail -30

Container metasploitable2 started
Container smb started
Container ftp started
SF:4\x20Oct\x202026\x2007:39:38\x20GMT\r\nConnection:\x20close\r\n\r\n")%r
SF:(RTSPRequest,52,"HTTP/1\.1\x20404\x20Not\x20Found\r\nDate:\x20Wed,\x201
SF:4\x20Oct\x202026\x2007:39:38\x20GMT\r\nConnection:\x20close\r\n\r\n")%r
SF:(FourOhFourRequest,52,"HTTP/1\.1\x20404\x20Not\x20Found\r\nDate:\x20Wed
SF:,\x2014\x20Oct\x202026\x2007:39:38\x20GMT\r\nConnection:\x20close\r\n\r
SF:\n")%r(Socks5,2F,"HTTP/1\.1\x20400\x20Bad\x20Request\r\nConnection:\x20
SF:close\r\n\r\n")%r(Socks4,2F,"HTTP/1\.1\x20400\x20Bad\x20Request\r\nConn
SF:ection:\x20close\r\n\r\n")%r(RPCCheck,2F,"HTTP/1\.1\x20400\x20Bad\x20Re
SF:quest\r\nConnection:\x20close\r\n\r\n")%r(DNSVersionBindReqTCP,2F,"HTTP
SF:/1\.1\x20400\x20Bad\x20Request\r\nConnection:\x20close\r\n\r\n")%r(DNSS
SF:tatusRequestTCP,2F,"HTTP/1\.1\x20400\x20Bad\x20Request\r\nConnection:\x
SF:20close\r\n\r\n")%r(Help,2F,"HTTP/1\.1\x20400\x20Bad\x20Request\r\nConn
SF:ection:\x20close\r\

In [ ]:
%%bash
# --- local exploit framework ---
docker exec redteam bash -lc 'msfconsole -q -x "version; exit"' | head -5
docker exec redteam bash -lc 'which nmap sqlmap hydra netexec responder' 2>/dev/null || true

 
 ****************************************************************************** 
 *                                                                            * 
 *               STARTING 6f116242-25f5-3681-9311-04f867f54e0c                * 
 *                                                                            * 
 
 ****************************************************************************** 
 *                                                                            * 
 *               STARTING 6f116242-25f5-3681-9311-04f867f54e0c                * 
 *                                                                            * 
 ****************************************************************************** 
 executing: bash


bash: line 1: msfconsole: command not found


In [ ]:
# --- example agent tasks ---
!nmap -sV -sC -p- -T4 localhost | tee /content/nmap_full.txt | tail -40
!enum4linux-ng -A localhost || true
!smbmap -H localhost -u user -p pass || true
!hydra -l user -P /usr/share/wordlists/rockyou.txt ssh://localhost -t 4 -f || true

SF:ess-Control-Allow-Methods:\x20GET,POST,OPTIONS\r\nAccess-Control-Allow-
SF:Headers:\x20ZAP-Header\r\nX-Frame-Options:\x20DENY\r\nX-XSS-Protection:
SF:\x201;\x20mode=block\r\nX-Content-Type-Options:\x20nosniff\r\nX-Clacks-
SF:Overhead:\x20GNU\x20Terry\x20Pratchett\r\nContent-Length:\x201420\r\nCo
SF:ntent-Type:\x20text/html\r\n\r\n<!DOCTYPE\x20html>\n<html\x20lang=\"en\
SF:">\n<head>\n\t<title>Welcome\x20to\x20the\x20HUD</title>\n\t<link\x20hr
SF:ef=\"tutorial\.css\"\x20rel=\"stylesheet\"\x20type=\"text/css\"\x20/>\n
SF:\t<script\x20src=\"Tutorial\.js\"></script>\x20\n</head>\n<body>\n<div\
SF:x20class=\"roundContainer\">\n\t<h1>Welcome\x20to\x20the\x20HUD</H1>\n\
SF:tThe\x20HUD\x20is\x20a\x20completely\x20new\x20way\x20to\x20interact\x2
SF:0with\x20ZAP\.<br>\n\tIt\x20overlays\x20security\x20information\x20on\x
SF:20top\x20of\x20the\x20application\x20you\x20are\x20test")%r(HTTPOptions
SF:,794,"HTTP/1\.1\x20200\x20OK\r\nPragma:\x20no-cache\r\nCache-Control:\x
SF:20no-cache,\x20no-stor

/bin/bash: line 1: enum4linux-ng: command not found


/usr/lib/python3/dist-packages/smbmap/smbmap.py:426: SyntaxWarning: invalid escape sequence '\p'
  stringbinding = 'ncacn_np:%s[\pipe\svcctl]' % remoteName
/usr/lib/python3/dist-packages/smbmap/smbmap.py:625: SyntaxWarning: invalid escape sequence '\*'
  tmp_bat_cmd = 'powershell -NoLogo -ExecutionPolicy bypass -Command " & {}Get-ChildItem {}\*.* -Recurse -Exclude *.dll,*.exe,*.msi,*.jpg,*.gif,*.bmp,*.png,*.mp3,*.wav | Select-String -Pattern \'{}\' | Select-Object -Unique Path | out-string -width 220{}" 2>nul > {}\{}.txt'.format('{', search_path, pattern, '}', tmp_dir, job_name)
/usr/lib/python3/dist-packages/smbmap/smbmap.py:632: SyntaxWarning: invalid escape sequence '\{'
  print('[+] Job {} started on {}, result will be stored at {}\{}.txt'.format(job_name, host, tmp_dir, job_name))
/usr/lib/python3/dist-packages/smbmap/smbmap.py:647: SyntaxWarning: invalid escape sequence '\{'
  file_path = '{}\{}.txt'.format(path, job)
/usr/lib/python3/dist-packages/smbmap/smbmap.py:665: SyntaxWar

/usr/lib/python3/dist-packages/impacket/version.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



    ________  ___      ___  _______   ___      ___       __         _______
   /"       )|"  \    /"  ||   _  "\ |"  \    /"  |     /""\       |   __ "\
  (:   \___/  \   \  //   |(. |_)  :) \   \  //   |    /    \      (. |__) :)
   \___  \    /\  \/.    ||:     \/   /\   \/.    |   /' /\  \     |:  ____/
    __/  \   |: \.        |(|  _  \  |: \.        |  //  __'  \    (|  /
   /" \   :) |.  \    /:  ||: |_)  :)|.  \    /:  | /   /  \   \  /|__/ \
  (_______/  |___|\__/|___|(_______/ |___|\__/|___|(___/    \___)(_______)
 -----------------------------------------------------------------------------
     SMBMap - Samba Share Enumerator | Shawn Evans - ShawnDEvans@gmail.com
                     https://github.com/ShawnDEvans/smbmap

[*] Detected 0 hosts serving SMB


Hydra v9.5 (c) 2023 by van Hauser/THC & David Maciejak - Please do not use in military or secret service organizations, or for illegal purposes (this is non-binding, these *** ignore laws and ethics anyway).

Hydra (https://github.com/vanhauser-thc/thc-hydra) starting at 2026-10-14 07:42:36
[ERROR] File for passwords not found: /usr/share/wordlists/rockyou.txt
